# 陨石图像分类项目

本项目包含三个核心步骤：
1. **背景去除** (`preprocess_bg.py`)  
2. **数据集类定义** (`dataset.py`)  
3. **训练与提交** (`main.py`)

下面将三个模块整合在一个 Notebook 中，按顺序执行即可完成数据预处理、模型训练和测试集预测。

## 环境准备
首先安装可能缺失的第三方库（如果尚未安装）。

In [ ]:
!pip install torch torchvision pandas pillow scikit-learn tqdm rembg

## 1. 背景去除预处理
使用 `rembg` 库移除训练集和测试集图像的背景，并统一转换为黑底 JPG 格式。

In [ ]:
import os
from PIL import Image
from rembg import remove
from tqdm import tqdm

def process_directory(input_dir, output_dir):
    """
    移除目录下所有图片的背景，保存为 JPG 格式。
    参数:
        input_dir: 原始图片文件夹路径
        output_dir: 输出抠图后图片的文件夹路径
    """
    if not os.path.exists(input_dir):
        print(f"⚠️ 找不到目录: {input_dir}")
        return

    os.makedirs(output_dir, exist_ok=True)
    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
    # 筛选出有效的图片文件
    filenames = [
        f for f in os.listdir(input_dir)
        if os.path.isfile(os.path.join(input_dir, f))
        and os.path.splitext(f)[1].lower() in valid_extensions
    ]

    print(f"📁 处理中: {input_dir} -> {output_dir}")
    for filename in tqdm(filenames, desc="RemBG"):
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, os.path.splitext(filename)[0] + ".jpg")

        # 如果已经存在则跳过，支持断点续跑
        if os.path.exists(output_path):
            continue

        try:
            img = Image.open(input_path).convert("RGB")
            subject = remove(img)  # 移除背景，返回 RGBA 图像
            # 如果抠图结果为空（没有主体），则跳过该图片
            if not subject.getbbox():
                continue

            # 将主体粘贴到黑色背景上
            bg = Image.new("RGB", subject.size, (0, 0, 0))
            bg.paste(subject, mask=subject.split()[3])  # 使用 alpha 通道作为 mask
            bg.save(output_path, format="JPEG", quality=95)
        except Exception as e:
            continue  # 忽略损坏或无法处理的图片

# 实际执行预处理
DATASET_ROOT = "./"  # 根据实际情况修改数据集根目录
process_directory(os.path.join(DATASET_ROOT, "train_images"),
                  os.path.join(DATASET_ROOT, "train_images_bg_removed"))
process_directory(os.path.join(DATASET_ROOT, "test_images"),
                  os.path.join(DATASET_ROOT, "test_images_bg_removed"))
print("\n✅ 预处理完成！")

## 2. 数据集类定义
自定义 `StoneDataset` 类，用于加载抠图后的图片，并根据划分模式（训练/验证/测试）提供数据。

In [1]:
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
import numpy as np

class StoneDataset(Dataset):
    """
    宝石图像数据集。
    参数:
        root: 数据集根目录
        split: 数据集划分，'train'/'val'/'test'
        transforms: 图像预处理/增强
        val_ratio: 验证集比例（仅对训练集有效）
        seed: 随机种子，确保划分可复现
    """
    def __init__(self, root, split="train", transforms=None, val_ratio=0.2, seed=42):
        self.root = root
        self.split = split
        self.transforms = transforms

        # 所有模式均使用抠图后的文件夹
        if split == "test":
            self.img_dir = os.path.join(root, "test_images_bg_removed")
        else:
            self.img_dir = os.path.join(root, "train_images_bg_removed")

        self.samples = []   # 图片路径列表
        self.labels = []    # 标签列表（测试集为 None）
        self.raw_ids = []   # 原始 CSV 中的 id 字符串，用于生成提交文件

        if split == "test":
            self._load_test_samples()
        else:
            self._load_train_val_samples(val_ratio, seed)

    def _load_train_val_samples(self, val_ratio, seed):
        """加载训练/验证集样本，内部进行固定划分"""
        csv_path = os.path.join(self.root, "train_labels.csv")
        df = pd.read_csv(csv_path)

        # 固定划分逻辑，保证每次运行结果一致
        indices = np.arange(len(df))
        state = np.random.RandomState(seed)
        state.shuffle(indices)

        val_size = int(len(df) * val_ratio)
        if self.split == "val":
            selected_indices = indices[:val_size]
        else:
            selected_indices = indices[val_size:]
        selected_df = df.iloc[selected_indices]

        for _, row in selected_df.iterrows():
            img_id = str(row["id"])
            # 抠图后统一为 .jpg 后缀
            img_path = os.path.join(self.img_dir, os.path.splitext(img_id)[0] + ".jpg")

            if os.path.exists(img_path):  # 只加载抠图成功的图片
                self.samples.append(img_path)
                self.labels.append(int(row["label"]))
                self.raw_ids.append(img_id)

    def _load_test_samples(self):
        """加载测试集样本，标签置为 None"""
        csv_path = os.path.join(self.root, "sample_submission.csv")
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            img_id = str(row["id"])
            img_path = os.path.join(self.img_dir, os.path.splitext(img_id)[0] + ".jpg")
            # 即使文件不存在也添加，防止提交时 id 对不上
            self.samples.append(img_path)
            self.labels.append(None)
            self.raw_ids.append(img_id)

    def __getitem__(self, index):
        img_path = self.samples[index]
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            # 如果图片丢失（例如抠图失败），返回纯黑图片避免崩溃
            image = Image.new("RGB", (224, 224), (0, 0, 0))

        if self.transforms:
            image = self.transforms(image)

        if self.split == "test":
            # 返回原始 ID，确保 submission 能正确映射
            return image, self.raw_ids[index]
        return image, self.labels[index]

    def __len__(self):
        return len(self.samples)

## 3. 训练与推理主程序
使用 ResNet-18 进行二分类训练，并在验证集上监控 F1 分数，最后对测试集进行预测并生成提交文件。

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, models
import pandas as pd
import os
from sklearn.metrics import f1_score
from tqdm import tqdm
# 注意：上面的数据集类定义必须在此行之前执行

def run_epoch(model, loader, criterion, optimizer, device, phase='train'):
    """
    运行一个 epoch 的训练或验证。
    返回: 平均损失, 宏平均 F1 分数
    """
    if phase == 'train':
        model.train()
    else:
        model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        with torch.set_grad_enabled(phase == 'train'):
            outputs = model(images)
            loss = criterion(outputs, labels)
            if phase == 'train':
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        running_loss += loss.item() * images.size(0)
        all_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_f1 = f1_score(all_labels, all_preds, average='macro')
    return running_loss / len(loader.dataset), epoch_f1

def main():
    DATASET_ROOT = "./"          # 数据集根目录
    BATCH_SIZE = 32
    LR = 0.0001
    NUM_EPOCHS = 12
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 图像预处理定义
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # 构建数据集和数据加载器
    train_ds = StoneDataset(DATASET_ROOT, split="train", transforms=train_transform)
    val_ds = StoneDataset(DATASET_ROOT, split="val", transforms=val_transform)
    test_ds = StoneDataset(DATASET_ROOT, split="test", transforms=val_transform)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    # 模型：在 ResNet-18 预训练权重上微调
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, 2)  # 二分类
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)

    # 训练循环，保存验证集 F1 最高的模型
    best_f1 = 0.0
    for epoch in range(NUM_EPOCHS):
        t_loss, t_f1 = run_epoch(model, train_loader, criterion, optimizer, device, 'train')
        v_loss, v_f1 = run_epoch(model, val_loader, criterion, None, device, 'val')

        print(f"Epoch {epoch+1}/{NUM_EPOCHS}: Train F1 {t_f1:.4f} | Val F1 {v_f1:.4f}")
        if v_f1 > best_f1:
            best_f1 = v_f1
            torch.save(model.state_dict(), "best_stone_model.pth")
            print(f"  ⭐ Best Model Saved (F1: {best_f1:.4f})")

    # ------------------- 测试集推理 -------------------
    print("\nStarting Test Inference...")
    model.load_state_dict(torch.load("best_stone_model.pth"))
    model.eval()

    results = {}  # key: 原始 id, value: 预测标签
    with torch.no_grad():
        for images, original_ids in tqdm(test_loader):
            outputs = model(images.to(device))
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            for oid, p in zip(original_ids, preds):
                results[oid] = int(p)

    # 生成提交文件
    sub = pd.read_csv(os.path.join(DATASET_ROOT, "sample_submission.csv"))
    sub["label"] = sub["id"].map(results).fillna(0).astype(int)

    print(f"Prediction distribution:\n{sub['label'].value_counts()}")
    os.makedirs("logs", exist_ok=True)
    sub.to_csv("logs/submission.csv", index=False)
    print("✅ Submission file saved to logs/submission.csv")

# 执行主程序
if __name__ == "__main__":
    main()

Epoch 1/12: Train F1 0.8691 | Val F1 0.9116
  ⭐ Best Model Saved (F1: 0.9116)
Epoch 2/12: Train F1 0.9270 | Val F1 0.8861
Epoch 3/12: Train F1 0.9530 | Val F1 0.8752
Epoch 4/12: Train F1 0.9587 | Val F1 0.9174
  ⭐ Best Model Saved (F1: 0.9174)
Epoch 5/12: Train F1 0.9666 | Val F1 0.9273
  ⭐ Best Model Saved (F1: 0.9273)
Epoch 6/12: Train F1 0.9732 | Val F1 0.8981
Epoch 7/12: Train F1 0.9747 | Val F1 0.9205
Epoch 8/12: Train F1 0.9730 | Val F1 0.9429
  ⭐ Best Model Saved (F1: 0.9429)
Epoch 9/12: Train F1 0.9759 | Val F1 0.9410
Epoch 10/12: Train F1 0.9855 | Val F1 0.9362
Epoch 11/12: Train F1 0.9808 | Val F1 0.9400
Epoch 12/12: Train F1 0.9845 | Val F1 0.9391

Starting Test Inference...


100%|██████████| 16/16 [00:34<00:00,  2.15s/it]

Prediction distribution:
label
0    261
1    250
Name: count, dtype: int64
✅ Submission file saved to logs/submission.csv


## 说明
- 确保目录结构为：
  - `./train_images/` 和 `./test_images/` 存放原始图片
  - `./train_labels.csv` 训练集标签文件
  - `./sample_submission.csv` 测试集样本提交模板
- 预处理会生成 `train_images_bg_removed/` 和 `test_images_bg_removed/`
- 训练好的模型保存为 `best_stone_model.pth`，最终预测文件在 `logs/submission.csv`